# Bagian C, sesi 2 — eksperimen tambahan untuk catatan reviewer yang belum tuntas

**Sesi 2.** Hasil sesi 1 (20 run) sudah tertanam di notebook ini, jadi **tidak perlu menambahkan Input** apa pun selain dataset.
Antrean sesi 2:
- `vit_s16` seed 2, 3, 4 (melengkapi ViT menjadi 5 seed, sama dengan model lain);
- EfficientNetB0 **tanpa augmentasi** seed 3 dan 4 (melengkapi menjadi 5 seed);
- EfficientNetB0 **+ SE r=8 tanpa augmentasi** seed 0 sampai 4 (menguji apakah kesimpulan tentang blok SE tetap sama pada konfigurasi pelatihan terbaik).

**Cara pakai**: GPU T4 x2, Internet ON, dataset *eye-diseases-classification* terpasang, lalu **Save Version → Save & Run All (Commit)**. Perkiraan 6 sampai 8 jam. Keluaran di tab Output, folder `hasil_C`.


### B0. Keras 2 dan deteksi GPU (jalankan pertama)

In [ ]:
import os, sys, subprocess, importlib.util
from importlib.metadata import version, PackageNotFoundError
try:
    _tf = tuple(int(v) for v in version('tensorflow').split('.')[:2])
except PackageNotFoundError:
    _tf = (0, 0)
if _tf >= (2, 16):                                # TF >= 2.16 bawaannya Keras 3 -> pakai tf-keras (Keras 2)
    os.environ['TF_USE_LEGACY_KERAS'] = '1'       # diwarisi juga oleh proses pekerja
    if importlib.util.find_spec('tf_keras') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', f'tf-keras~={_tf[0]}.{_tf[1]}.0'], check=True)
    import tf_keras; print('Keras 2 aktif (tf_keras', tf_keras.__version__ + ')')
else:
    print('TensorFlow', '.'.join(map(str, _tf)), '-> sudah Keras 2')

# Hitung GPU, lalu SEMBUNYIKAN dari proses notebook ini: GPU hanya dipakai oleh proses pekerja (satu pekerja per GPU).
try:
    N_GPU = len([l for l in subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.splitlines() if l.startswith('GPU ')])
except Exception:
    N_GPU = 0
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
print('GPU terdeteksi:', N_GPU)


### Konfigurasi

In [ ]:
import json, time, glob, math, itertools
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
_CANDS = ['/kaggle/input/eye-diseases-classification/dataset',
          '/kaggle/input/datasets/gunavenkatdoddi/eye-diseases-classification/dataset',
          'dataset']
DATASET_DIR = next((d for d in _CANDS if os.path.isdir(d)), None)
if DATASET_DIR is None:   # cari folder dataset di mana pun di /kaggle/input (nama folder bisa berbeda)
    _cari = [os.path.dirname(d) for d in glob.glob('/kaggle/input/**/glaucoma', recursive=True)
             if os.path.isdir(os.path.join(os.path.dirname(d), 'cataract'))]
    DATASET_DIR = _cari[0] if _cari else None
if DATASET_DIR is None:
    raise RuntimeError('DATASET TIDAK TERPASANG. Di editor: panel kanan -> Add Input -> cari "eye-diseases-classification" '
                       '(gunavenkatdoddi) -> Add. Setelah itu baru Save Version -> Save & Run All.')
OUT_DIR = '/kaggle/working/hasil_C' if os.path.isdir('/kaggle/working') else 'hasil_C'

# ---- konfigurasi pelatihan: HARUS sama dengan Bagian A di notebook utama
SEED, IMG_SIZE, BATCH_SIZE, EPOCHS, VAL_SPLIT, SE_RATIO = 123, (224, 224), 32, 150, 0.10, 8
USE_STOP_CRITERION    = True     # tidak dipakai di Bagian B (run_experiment memanggil use_stop=False)
BN_BACKBONE_INFERENCE = True
LR_PER_ITERASI        = False
PANTAU                = 'val_accuracy'   # besaran untuk early stopping + bobot terbaik. 'val_loss' = resep lama (Bagian A)
PATIENCE              = 30

# ---- apa yang dijalankan. Urutan: semua varian untuk seed pertama dulu, lalu seed berikutnya,
#      sehingga kalau waktu habis tabel tetap lengkap (hanya jumlah seed-nya lebih sedikit).
RUN_VARIANTS = [('vit_s16', 'none', 8, 'dasar'), ('efficientnetb0', 'none', 8, 'aug_tanpa'), ('efficientnetb0', 'se', 8, 'aug_tanpa')]
RUN_SEEDS    = [0, 1, 2, 3, 4]  # run yang sudah selesai di sesi 1 (tertanam di sel C2) otomatis dilewati

# ---- percepatan
N_PARALEL       = None    # None = satu pekerja per GPU (T4 x2 -> 2 run sekaligus). Isi 1 untuk berurutan.
BATAS_JAM       = 11.2    # semua run harus SELESAI sebelum jam ini (batas sesi Kaggle 12 jam). Run baru hanya dimulai bila perkiraan terlamanya masih muat.
MAKS_MENIT      = {'efficientnetb0': 63, 'vit_s16': 215, 'aug_tanpa': 55, 'res384': 190}   # lama terpanjang satu run 150 epoch di T4 (perkiraan; res384 = 3x piksel)
UJI_CEPAT       = False   # True: uji alur beberapa menit (2 run, 2 epoch, sedikit citra). Jalankan sekali dulu sebelum run panjang.

os.makedirs(OUT_DIR, exist_ok=True)
T_MULAI = time.time()
print('DATASET_DIR :', DATASET_DIR); print('OUT_DIR     :', os.path.abspath(OUT_DIR))


### B1. Data, duplikat, dan pembagian 70:15:15 (identik dengan notebook utama)

In [ ]:
import numpy as np, pandas as pd
from PIL import Image
import tensorflow as tf
keras = tf.keras
from sklearn.model_selection import train_test_split

# --- urutan berkas identik dengan notebook utama (seed 123)
common = dict(validation_split=VAL_SPLIT, seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
              shuffle=True, label_mode='int')
train_raw = keras.utils.image_dataset_from_directory(DATASET_DIR, subset='training', **common)
val_raw   = keras.utils.image_dataset_from_directory(DATASET_DIR, subset='validation', **common)
class_names = train_raw.class_names
NUM_CLASSES = len(class_names)
CLS_ID = {c: i for i, c in enumerate(class_names)}
GLAUCOMA = CLS_ID['glaucoma']; NORMAL = CLS_ID['normal']

train_files = list(train_raw.file_paths); val_files = list(val_raw.file_paths)
assert not set(train_files) & set(val_files), 'ada berkas yang muncul di latih dan validasi!'
lbl = lambda p: os.path.basename(os.path.dirname(p))
split_df = pd.DataFrame({'file': train_files + val_files,
                         'kelas': [lbl(p) for p in train_files + val_files],
                         'subset': ['latih'] * len(train_files) + ['validasi'] * len(val_files)})
print(pd.crosstab(split_df.kelas, split_df.subset, margins=True, margins_name='Total'))

# --- deteksi duplikat pHash (sama dengan A3)
from scipy.fftpack import dct
def phash(path, hash_size=8, highfreq=4):
    img = Image.open(path).convert('L').resize((hash_size * highfreq,) * 2, Image.LANCZOS)
    a = np.asarray(img, dtype=np.float32)
    d = dct(dct(a, axis=0, norm='ortho'), axis=1, norm='ortho')[:hash_size, :hash_size]
    bits = (d > np.median(d)).flatten()
    return np.packbits(bits).view('>u8')[0]

rows = []
for p in split_df.file:
    with Image.open(p) as im:
        w, h = im.size
    rows.append((w, h, os.path.splitext(p)[1].lower(), phash(p)))
_meta = pd.DataFrame(rows, index=split_df.index, columns=['lebar', 'tinggi', 'ekstensi', 'phash'])
for _c in _meta.columns: split_df[_c] = _meta[_c]

H = split_df.phash.values.astype(np.uint64)
def hamming(a, B):
    x = np.bitwise_xor(B, a)
    return np.unpackbits(x.view(np.uint8).reshape(-1, 8), axis=1).sum(1)
THR = 4   # jarak Hamming <= 4 dari 64 bit dianggap duplikat/hampir duplikat
pairs = []
for i in range(len(H)):
    d = hamming(H[i], H[i + 1:])
    for j in np.nonzero(d <= THR)[0]:
        pairs.append((i, i + 1 + j, int(d[j])))
dup = pd.DataFrame(pairs, columns=['i', 'j', 'hamming'])
if len(dup):
    dup['kelas_i'] = split_df.kelas.values[dup.i]; dup['kelas_j'] = split_df.kelas.values[dup.j]
    dup['subset_i'] = split_df.subset.values[dup.i]; dup['subset_j'] = split_df.subset.values[dup.j]
    dup['file_i'] = split_df.file.values[dup.i]; dup['file_j'] = split_df.file.values[dup.j]
print('Pasangan (hampir) duplikat:', len(dup))

# --- kelompok duplikat (union-find) dan pembagian 70:15:15 terstratifikasi (sama dengan B1)
parent = list(range(len(split_df)))
def find(a):
    while parent[a] != a: parent[a] = parent[parent[a]]; a = parent[a]
    return a
for i, j in zip(dup.i if len(dup) else [], dup.j if len(dup) else []): parent[find(i)] = find(j)
split_df['grup'] = [find(i) for i in range(len(split_df))]
konflik = split_df.groupby('grup').kelas.nunique(); konflik = konflik[konflik > 1].index
print('Grup duplikat dengan label berbeda (dibuang dari Bagian B):', len(konflik))
dfB = split_df[~split_df.grup.isin(konflik)].copy()
grp = dfB.groupby('grup').agg(kelas=('kelas', 'first')).reset_index()
g_tr, g_tmp = train_test_split(grp, test_size=0.30, stratify=grp.kelas, random_state=SEED)
g_va, g_te = train_test_split(g_tmp, test_size=0.50, stratify=g_tmp.kelas, random_state=SEED)
dfB['subset'] = dfB.grup.map({**{g: 'latih' for g in g_tr.grup}, **{g: 'validasi' for g in g_va.grup}, **{g: 'uji' for g in g_te.grup}})
dfB.to_csv(f'{OUT_DIR}/pembagian_70_15_15.csv', index=False)
print(pd.crosstab(dfB.kelas, dfB.subset, margins=True))
P = {s: dfB.file[dfB.subset == s].tolist() for s in ['latih', 'validasi', 'uji']}
if UJI_CEPAT:
    P = {s: v[::max(1, len(v) // n)][:n] for (s, v), n in zip(P.items(), [96, 48, 48])}
    print('UJI_CEPAT:', {s: len(v) for s, v in P.items()})


### Pustaka pekerja (ditulis ke `fundus_lib.py`)

In [ ]:
%%writefile fundus_lib.py
# Pekerja Bagian B: melatih SATU konfigurasi pada SATU GPU. Dipanggil oleh sel "B2" sebagai proses terpisah:
#   python fundus_lib.py <state.json> <job.json> <hasil.json>
# Definisi model, augmentasi, callback, dan metrik disalin apa adanya dari notebook utama.
import os, sys, json, time, math
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import numpy as np
import tensorflow as tf
keras = tf.keras
for _g in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(_g, True)
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize

ST = json.load(open(sys.argv[1], encoding='utf-8'))
globals().update(ST['konfig'])            # SEED, IMG_SIZE, BATCH_SIZE, EPOCHS, SE_RATIO, USE_STOP_CRITERION, BN_BACKBONE_INFERENCE, LR_PER_ITERASI, OUT_DIR, MIXED_PRECISION, PANTAU, PATIENCE
IMG_SIZE = tuple(IMG_SIZE)
JOB = json.load(open(sys.argv[2], encoding='utf-8')) if len(sys.argv) > 2 else {}
VARIAN = JOB.get('varian', 'dasar')
if VARIAN == 'res384': IMG_SIZE = (384, 384)
class_names = ST['class_names']; NUM_CLASSES = len(class_names); CLS_ID = {c: i for i, c in enumerate(class_names)}
P = ST['P']
lbl = lambda p: os.path.basename(os.path.dirname(p))
AUTOTUNE = tf.data.AUTOTUNE
NOISE_STD = 0.05
L = keras.layers
if MIXED_PRECISION:
    keras.mixed_precision.set_global_policy('mixed_float16')

def load_img(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE, method='bilinear')     # sama dengan image_dataset_from_directory
    img.set_shape(IMG_SIZE + (3,))
    if VARIAN == 'tepi': img = img * MASK_TEPI      # hanya tepi citra yang terlihat (kontrol jalan pintas, Reviewer A#4)
    return img

_yy, _xx = np.mgrid[0:IMG_SIZE[0], 0:IMG_SIZE[1]].astype('float32')
_r = np.sqrt((_yy - (IMG_SIZE[0] - 1) / 2) ** 2 + (_xx - (IMG_SIZE[1] - 1) / 2) ** 2) / (IMG_SIZE[1] / 2)
MASK_TEPI = tf.constant((_r > 0.85).astype('float32')[..., None])

def ds_from_paths(paths, training=False, seed=SEED):
    labels = [CLS_ID[lbl(p)] for p in paths]
    ds = tf.data.Dataset.from_tensor_slices((list(paths), labels))
    if IMG_SIZE[0] > 224:   # hemat RAM: simpan cache sebagai uint8 (selisih pembulatan < 0,5 tingkat keabuan)
        ds = ds.map(lambda p, y: (tf.cast(tf.round(load_img(p)), tf.uint8), tf.one_hot(y, NUM_CLASSES)), num_parallel_calls=AUTOTUNE).cache()
        ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.map(lambda p, y: (load_img(p), tf.one_hot(y, NUM_CLASSES)), num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(1000, seed=seed, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

@keras.utils.register_keras_serializable(package='fundus')   # keras.utils ada di Keras 2 dan 3 (keras.saving tidak dimuat lewat tf.keras di Keras 3)
class RandomCropResize(keras.layers.Layer):
    """Potong acak crop×crop lalu ubah kembali ke ukuran semula; HANYA saat pelatihan."""
    def __init__(self, crop=200, out=224, seed=None, **kw):
        super().__init__(**kw); self.crop, self.out, self.seed = crop, out, seed
    def call(self, x, training=None):
        if not training:
            return x
        b = tf.shape(x)[0]
        x = tf.image.random_crop(x, tf.stack([b, self.crop, self.crop, 3]), seed=self.seed)
        return tf.image.resize(x, (self.out, self.out))
    def get_config(self):
        return {**super().get_config(), 'crop': self.crop, 'out': self.out, 'seed': self.seed}

def make_augmentation(seed=SEED):
    S = IMG_SIZE[0]
    lap = [('crop', RandomCropResize(int(round(S * 200 / 224)), S, seed=seed)),
           ('flip', keras.layers.RandomFlip('horizontal', seed=seed)),
           ('rot', keras.layers.RandomRotation(0.1, seed=seed)),          # faktor 0,1 = ±36°
           ('zoom', keras.layers.RandomZoom(0.1, seed=seed)),
           ('kontras', keras.layers.RandomContrast(0.1, seed=seed)),
           ('cerah', keras.layers.RandomBrightness(0.1, seed=seed)),
           ('geser', keras.layers.RandomTranslation(0.1, 0.1, seed=seed)),
           ('derau', keras.layers.GaussianNoise(NOISE_STD * 255 if VARIAN == 'aug_derau_benar' else NOISE_STD, seed=seed))]
    buang = {'aug_tanpa': {n for n, _ in lap}, 'aug_tanpa_crop': {'crop'}, 'aug_tanpa_flip': {'flip'}}.get(VARIAN, set())
    lap = [l for n, l in lap if n not in buang]
    return keras.Sequential(lap or [keras.layers.Activation('linear')], name='augmentasi')

def attention_block(x, kind='se', ratio=8):
    C = x.shape[-1]
    if kind == 'none':
        return x
    if kind == 'se':
        s = L.GlobalAveragePooling2D(name='se_squeeze')(x)
        s = L.Reshape((1, 1, C), name='se_reshape')(s)
        s = L.Dense(C // ratio, activation='relu', name='se_fc1')(s)
        s = L.Dense(C, activation='sigmoid', name='se_fc2')(s)
        return L.Multiply(name='se_scale')([x, s])
    if kind == 'eca':           # Wang dkk. (ECA-Net), k adaptif
        k = int(abs((math.log2(C) + 1) / 2)); k = k if k % 2 else k + 1
        s = L.GlobalAveragePooling2D(name='eca_gap')(x)
        s = L.Reshape((C, 1), name='eca_reshape1')(s)
        s = L.Conv1D(1, k, padding='same', use_bias=False, name='eca_conv')(s)
        s = L.Activation('sigmoid', name='eca_sigmoid')(s)
        s = L.Reshape((1, 1, C), name='eca_reshape2')(s)
        return L.Multiply(name='eca_scale')([x, s])
    if kind == 'cbam':          # Woo dkk. (CBAM): atensi kanal lalu spasial
        fc1 = L.Dense(C // ratio, activation='relu', name='cbam_fc1'); fc2 = L.Dense(C, name='cbam_fc2')
        a = fc2(fc1(L.GlobalAveragePooling2D(name='cbam_gap')(x)))
        m = fc2(fc1(L.GlobalMaxPooling2D(name='cbam_gmp')(x)))
        ca = L.Activation('sigmoid', name='cbam_ch_sig')(L.Add(name='cbam_ch_add')([a, m]))
        x = L.Multiply(name='cbam_ch_scale')([x, L.Reshape((1, 1, C), name='cbam_ch_reshape')(ca)])
        avg = L.Lambda(lambda t: tf.reduce_mean(t, -1, keepdims=True), name='cbam_sp_mean')(x)
        mx = L.Lambda(lambda t: tf.reduce_max(t, -1, keepdims=True), name='cbam_sp_max')(x)
        sa = L.Conv2D(1, 7, padding='same', activation='sigmoid', name='cbam_sp_conv')(L.Concatenate(name='cbam_sp_cat')([avg, mx]))
        return L.Multiply(name='cbam_sp_scale')([x, sa])
    raise ValueError(kind)

def make_backbone(name):
    if name == 'efficientnetb0':
        return keras.applications.EfficientNetB0(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,)), None
    if name == 'resnet50':
        return keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,)), keras.applications.resnet50.preprocess_input
    if name == 'mobilenetv2':
        return keras.applications.MobileNetV2(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,)), keras.applications.mobilenet_v2.preprocess_input
    if name == 'vit_s16':     # ViT-S/16 pra-latih ImageNet (TF-Hub); keluaran berupa vektor ciri 384 dimensi
        import tensorflow_hub as hub
        return hub.KerasLayer('https://tfhub.dev/sayakpaul/vit_s16_fe/1', trainable=True, name='vit_s16'), (lambda t: t / 127.5 - 1.0)
    raise ValueError(name)

def build_model(backbone='efficientnetb0', attention='se', ratio=SE_RATIO, seed=SEED):
    base, prep = make_backbone(backbone)
    base.trainable = True                      # seluruh lapisan di-fine-tune
    inputs = L.Input(IMG_SIZE + (3,), name='citra')
    x = make_augmentation(seed)(inputs)        # PERBAIKAN #1: keluaran augmentasi dipakai
    if prep is not None:                        # EfficientNet Keras sudah memuat rescaling+normalisasi
        x = L.Lambda(prep, name='preprocess')(x)
    # PERBAIKAN #2: training tidak dikunci ke True. PERBAIKAN #9: BN backbone dalam mode inferensi (bobot tetap dilatih)
    if backbone == 'vit_s16':
        x = base(x)                               # transformer: tidak ada BatchNorm dan keluarannya sudah berupa vektor
    else:
        x = base(x, training=False) if BN_BACKBONE_INFERENCE else base(x)
        x = attention_block(x, attention, ratio)
        x = L.GlobalAveragePooling2D(name='head_gap')(x)
    x = L.BatchNormalization(name='head_bn1')(x)
    x = L.Dense(128, activation='relu', name='head_fc')(x)
    x = L.BatchNormalization(name='head_bn2')(x)
    x = L.Dropout(0.3, name='head_dropout')(x)
    out = L.Dense(NUM_CLASSES, activation='softmax', name='keluaran', dtype='float32')(x)   # float32 agar aman bila mixed precision
    return keras.Model(inputs, out, name=f'{backbone}_{attention}{ratio if attention in ("se","cbam") else ""}'), base

def cyclic_lr(epoch, lr=None, base_lr=1e-5, max_lr=1e-3, step_size=2000):
    cycle = np.floor(1 + epoch / (2 * step_size))
    x = np.abs(epoch / step_size - 2 * cycle + 1)
    return float(base_lr + (max_lr - base_lr) * max(0, 1 - x))

class CyclicLRIter(keras.callbacks.Callback):
    """Persamaan (4) dan (5) dengan t = nomor iterasi (batch), bukan nomor epoch."""
    def on_train_begin(self, logs=None): self.it = 0
    def on_train_batch_begin(self, batch, logs=None):
        keras.backend.set_value(self.model.optimizer.learning_rate, cyclic_lr(self.it)); self.it += 1
    def on_epoch_end(self, epoch, logs=None):
        if logs is not None: logs['lr'] = float(keras.backend.get_value(self.model.optimizer.learning_rate))

class StopCriterion(keras.callbacks.Callback):
    def __init__(self): super().__init__(); self.fired_epoch = None
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        if logs.get('val_accuracy', 0) >= 0.95 and logs.get('val_loss', 9) < 0.13:
            self.fired_epoch = epoch + 1; self.model.stop_training = True

class Timer(keras.callbacks.Callback):
    def on_train_begin(self, logs=None): self.t0 = time.time()
    def on_train_end(self, logs=None): self.seconds = time.time() - self.t0

class BestKeeper(keras.callbacks.Callback):
    """Simpan bobot epoch terbaik di memori dan pasang kembali di akhir pelatihan, juga bila early stopping TIDAK terpicu
    (EarlyStopping(restore_best_weights) tidak menjamin itu saat epoch habis)."""
    def __init__(self, monitor):
        super().__init__(); self.monitor = monitor; self.sign = -1 if 'loss' in monitor else 1
        self.best = -np.inf; self.best_epoch = 0; self.w = None
    def on_epoch_end(self, epoch, logs=None):
        v = self.sign * float((logs or {}).get(self.monitor, -np.inf * self.sign))
        if v > self.best: self.best, self.best_epoch, self.w = v, epoch + 1, self.model.get_weights()
    def on_train_end(self, logs=None):
        if self.w is not None: self.model.set_weights(self.w)

def make_callbacks(ckpt_path, use_stop=USE_STOP_CRITERION, log_path=None):
    stop, timer = StopCriterion(), Timer()
    es = keras.callbacks.EarlyStopping(monitor=PANTAU, mode=('min' if 'loss' in PANTAU else 'max'), patience=PATIENCE)
    es.keeper = BestKeeper(PANTAU)
    cbs = [CyclicLRIter() if LR_PER_ITERASI else keras.callbacks.LearningRateScheduler(cyclic_lr), es, es.keeper, timer]
    if use_stop: cbs.insert(0, stop)
    if log_path: cbs.append(keras.callbacks.CSVLogger(log_path))
    return cbs, stop, es, timer

def predict_paths(m, paths):
    ds = ds_from_paths(paths)
    probs = m.predict(ds, verbose=0)
    y = np.array([CLS_ID[lbl(p)] for p in paths])
    return y, probs

def confusion(y, yp, k):
    cm = np.zeros((k, k), int); np.add.at(cm, (y, yp), 1); return cm

def per_class(cm):
    n = cm.sum(); out = []
    for i in range(len(cm)):
        tp = cm[i, i]; fn = cm[i].sum() - tp; fp = cm[:, i].sum() - tp; tn = n - tp - fn - fp
        prec = tp / (tp + fp) if tp + fp else 0.0; rec = tp / (tp + fn) if tp + fn else 0.0
        out.append(dict(tp=tp, fp=fp, fn=fn, tn=tn, presisi=prec, sensitivitas=rec,
                        spesifisitas=tn / (tn + fp) if tn + fp else 0.0,
                        f1=2 * prec * rec / (prec + rec) if prec + rec else 0.0, support=tp + fn))
    return out

def kappa(cm):
    n = cm.sum(); po = np.trace(cm) / n; pe = (cm.sum(0) * cm.sum(1)).sum() / n**2
    return (po - pe) / (1 - pe)

def summary_metrics(y, yp, k):
    cm = confusion(y, yp, k); pc = per_class(cm); sup = np.array([d['support'] for d in pc])
    macro = {m: np.mean([d[m] for d in pc]) for m in ['presisi', 'sensitivitas', 'spesifisitas', 'f1']}
    weighted = {m: np.average([d[m] for d in pc], weights=sup) for m in ['presisi', 'sensitivitas', 'spesifisitas', 'f1']}
    return dict(cm=cm, per_class=pc, macro=macro, weighted=weighted, akurasi=np.trace(cm) / cm.sum(), kappa=kappa(cm))

_make_aug = make_augmentation
def make_augmentation(seed=SEED):
    # augmentasi selalu float32 (RandomContrast tidak punya kernel float16); hanya backbone dan head yang float16
    if not MIXED_PRECISION: return _make_aug(seed)
    keras.mixed_precision.set_global_policy('float32'); aug = _make_aug(seed)
    keras.mixed_precision.set_global_policy('mixed_float16'); return aug

class EpochLog(keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None): self.t = time.time()
    def on_epoch_end(self, epoch, logs=None):
        l = logs or {}
        print(f"epoch {epoch+1}/{EPOCHS} loss {l.get('loss', 0):.4f} acc {l.get('accuracy', 0):.4f} "
              f"val_loss {l.get('val_loss', 0):.4f} val_acc {l.get('val_accuracy', 0):.4f} ({time.time()-self.t:.0f}s)", flush=True)

def run_experiment(backbone, attention, ratio, seed):
    keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed)
    m, _ = build_model(backbone, attention, ratio, seed=seed)
    m.compile(optimizer=keras.optimizers.Adam(1e-3, clipnorm=1.0), loss='categorical_crossentropy', metrics=['accuracy'])
    ytr = np.array([CLS_ID[lbl(p)] for p in P['latih']]); cnt = np.bincount(ytr, minlength=NUM_CLASSES)
    cw = {i: len(ytr) / (NUM_CLASSES * c) for i, c in enumerate(cnt)}
    ck = f'{OUT_DIR}/tmp_{backbone}_{attention}{ratio}_s{seed}.keras'
    cbs, _, es, timer = make_callbacks(ck, use_stop=False)
    h = m.fit(ds_from_paths(P['latih'], training=True, seed=seed), validation_data=ds_from_paths(P['validasi']),
              epochs=EPOCHS, class_weight=cw, callbacks=cbs + [EpochLog()], verbose=0)
    yt, pr = predict_paths(m, P['uji']); pr = pr.astype('float32'); s = summary_metrics(yt, pr.argmax(1), NUM_CLASSES)
    ybt = label_binarize(yt, classes=range(NUM_CLASSES))
    out = dict(backbone=backbone, atensi=attention, rasio=ratio, seed=seed, varian=VARIAN, ukuran_masukan=IMG_SIZE[0], param=m.count_params(),
               epoch=len(h.history['loss']), epoch_terbaik=int(es.keeper.best_epoch), val_akurasi_terbaik=float(max(h.history['val_accuracy'])), menit=timer.seconds / 60,
               akurasi=s['akurasi'], macro_f1=s['macro']['f1'], kappa=s['kappa'], auc_micro=roc_auc_score(ybt, pr, average='micro'),
               **{f'sens_{c}': d['sensitivitas'] for c, d in zip(class_names, s['per_class'])})
    np.save(f'{OUT_DIR}/prob_uji_{backbone}_{VARIAN}_s{seed}.npy', pr)
    if os.path.exists(ck): os.remove(ck)
    return out

if __name__ == '__main__':
    job = JOB
    print('GPU terlihat:', [d.name for d in tf.config.list_physical_devices('GPU')] or 'CPU saja', '| job:', job, flush=True)
    row = run_experiment(job['backbone'], job['atensi'], job['rasio'], job['seed'])
    json.dump(row, open(sys.argv[3], 'w', encoding='utf-8'), default=float)
    print('SELESAI', {k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()}, flush=True)


### Hasil model dasar dari Bagian B (pembanding)


In [ ]:
CSV_DASAR = """backbone,atensi,rasio,seed,konfig,param,epoch,epoch_terbaik,val_akurasi_terbaik,menit,akurasi,macro_f1,kappa,auc_micro,sens_cataract,sens_diabetic_retinopathy,sens_glaucoma,sens_normal
efficientnetb0,none,8,0,bn1_lriter0_ep150_val_accuracy_p30,4219687.0,104.0,74.0,0.8946428298950195,41.921940747896834,0.8792792792792793,0.8820422066291519,0.8384363716941043,0.9833281930579229,0.9606299212598424,0.922077922077922,0.8582677165354331,0.782312925170068
mobilenetv2,none,8,0,bn1_lriter0_ep150_val_accuracy_p30,2428100.0,126.0,96.0,0.8785714507102966,31.959766288598377,0.8900900900900901,0.8903347617024497,0.8529660851324413,0.9801552904255608,0.9606299212598424,0.974025974025974,0.8818897637795275,0.7482993197278912
resnet50,none,8,0,bn1_lriter0_ep150_val_accuracy_p30,23859204.0,102.0,72.0,0.8999999761581421,64.66553745269775,0.8810810810810811,0.8799831393200297,0.8410142449153204,0.9774217460703948,0.9763779527559056,0.9545454545454546,0.8346456692913385,0.7619047619047619
efficientnetb0,none,8,1,bn1_lriter0_ep150_val_accuracy_p30,4219687.0,140.0,110.0,0.9107142686843872,55.49663483301799,0.8972972972972973,0.8984064066881404,0.8627167630057804,0.9789638286935584,0.968503937007874,0.9090909090909092,0.8661417322834646,0.8503401360544217
mobilenetv2,none,8,1,bn1_lriter0_ep150_val_accuracy_p30,2428100.0,100.0,70.0,0.8535714149475098,25.35594440698624,0.8468468468468469,0.848062691982396,0.7954675522874683,0.9709644779915052,0.8582677165354331,0.8701298701298701,0.9133858267716536,0.7551020408163265
resnet50,none,8,1,bn1_lriter0_ep150_val_accuracy_p30,23859204.0,129.0,99.0,0.8982142806053162,81.0705362478892,0.8756756756756757,0.8772543712319745,0.8338979228023303,0.9823174525877229,0.9448818897637796,0.8831168831168831,0.8661417322834646,0.8163265306122449
efficientnetb0,none,8,2,bn1_lriter0_ep150_val_accuracy_p30,4219687.0,150.0,137.0,0.9089285731315612,59.91405316988627,0.8828828828828829,0.8830113611471571,0.843537208434895,0.9824992560127696,0.968503937007874,0.935064935064935,0.889763779527559,0.7482993197278912
resnet50,none,8,2,bn1_lriter0_ep150_val_accuracy_p30,23859204.0,93.0,63.0,0.8982142806053162,59.180640856424965,0.8864864864864865,0.8853476486442207,0.8483981321297104,0.9818034250466684,0.968503937007874,0.9285714285714286,0.8346456692913385,0.8163265306122449
mobilenetv2,none,8,2,bn1_lriter0_ep150_val_accuracy_p30,2428100.0,123.0,93.0,0.8678571581840515,30.81873476902644,0.8522522522522522,0.8540387457331866,0.8021312950813256,0.9721667613559504,0.8503937007874016,0.9285714285714286,0.8661417322834646,0.7619047619047619
efficientnetb0,none,8,3,bn1_lriter0_ep150_val_accuracy_p30,4219687.0,129.0,99.0,0.8946428298950195,50.201711642742154,0.8972972972972973,0.8985708914714567,0.8628245099016986,0.9857879501122745,0.9606299212598425,0.8896103896103896,0.905511811023622,0.8435374149659864
mobilenetv2,none,8,3,bn1_lriter0_ep150_val_accuracy_p30,2428100.0,150.0,127.0,0.8928571343421936,34.8487802028656,0.8684684684684685,0.8678708071187333,0.8240123362943336,0.9800102805508211,0.984251968503937,0.9545454545454546,0.8110236220472441,0.7278911564625851
resnet50,none,8,3,bn1_lriter0_ep150_val_accuracy_p30,23859204.0,91.0,61.0,0.8910714387893677,52.778939350446066,0.8774774774774775,0.8791777201788344,0.8360855100285786,0.9754884614344075,0.952755905511811,0.935064935064935,0.8740157480314961,0.7551020408163265
efficientnetb0,none,8,4,bn1_lriter0_ep150_val_accuracy_p30,4219687.0,150.0,140.0,0.8999999761581421,57.51932620207469,0.8900900900900901,0.8918329962617952,0.8530834891966134,0.9775045315585855,0.968503937007874,0.8961038961038961,0.8582677165354331,0.8435374149659864
mobilenetv2,none,8,4,bn1_lriter0_ep150_val_accuracy_p30,2428100.0,138.0,108.0,0.887499988079071,32.21003271738688,0.863063063063063,0.8628125130999362,0.8168634942688433,0.9763190217244271,0.9763779527559056,0.948051948051948,0.8582677165354331,0.6802721088435374
resnet50,none,8,4,bn1_lriter0_ep150_val_accuracy_p30,23859204.0,114.0,84.0,0.8999999761581421,65.67034946680069,0.8900900900900901,0.8904249184071239,0.8529156203567736,0.9804247490733977,0.905511811023622,0.948051948051948,0.84251968503937,0.8571428571428571"""
print("Run pembanding tertanam:", len(CSV_DASAR.splitlines()) - 1)


### B2. Jalankan antrean: satu proses pekerja per GPU

Log tiap run ada di `hasil/log_B/<nama>.log`. Baris `...` adalah status berkala (epoch terakhir tiap pekerja).

In [ ]:
KONFIG = dict(SEED=SEED, IMG_SIZE=list(IMG_SIZE), BATCH_SIZE=BATCH_SIZE, EPOCHS=(2 if UJI_CEPAT else EPOCHS), SE_RATIO=SE_RATIO,
              USE_STOP_CRITERION=USE_STOP_CRITERION, BN_BACKBONE_INFERENCE=BN_BACKBONE_INFERENCE, LR_PER_ITERASI=LR_PER_ITERASI,
              OUT_DIR=OUT_DIR, MIXED_PRECISION=False, PANTAU=PANTAU, PATIENCE=PATIENCE)   # mixed precision diuji: loss NaN pada EfficientNet -> tidak dipakai
TAG = f'bn{int(BN_BACKBONE_INFERENCE)}_lriter{int(LR_PER_ITERASI)}_ep{KONFIG["EPOCHS"]}_{PANTAU}_p{PATIENCE}'   # run dengan konfigurasi lain tidak dianggap selesai
B_CSV = f'{OUT_DIR}/' + ('eksperimen_C_ujicepat.csv' if UJI_CEPAT else 'eksperimen_C.csv')
LOG_DIR = f'{OUT_DIR}/log_C'; os.makedirs(LOG_DIR, exist_ok=True)
STATE = f'{OUT_DIR}/state_C.json'
json.dump(dict(konfig=KONFIG, class_names=class_names, P=P), open(STATE, 'w', encoding='utf-8'))

# --- lanjutkan dari sesi sebelumnya: CSV lokal + CSV dari keluaran notebook lama yang ditambahkan sebagai Input
KEY = ['backbone', 'atensi', 'rasio', 'varian', 'seed', 'konfig']
CSV_SESI1 = """backbone,atensi,rasio,varian,seed,konfig,ukuran_masukan,param,epoch,epoch_terbaik,val_akurasi_terbaik,menit,akurasi,macro_f1,kappa,auc_micro,sens_cataract,sens_diabetic_retinopathy,sens_glaucoma,sens_normal
efficientnetb0,none,8,tepi,0,bn1_lriter0_ep150_val_accuracy_p30,224.0,4219687.0,150.0,140.0,0.8428571224212646,55.40050211350123,0.8684684684684685,0.8645347593582887,0.8243990603410165,0.9724080837594351,0.84251968503937,0.987012987012987,0.8346456692913385,0.7959183673469388
efficientnetb0,none,8,aug_tanpa,0,bn1_lriter0_ep150_val_accuracy_p30,224.0,4219687.0,56.0,26.0,0.9267857074737549,21.019712356726327,0.9387387387387387,0.9366863565520624,0.9181848926041224,0.9914736357979602,0.952755905511811,0.9935064935064936,0.9212598425196851,0.8843537414965986
efficientnetb0,none,8,res384,0,bn1_lriter0_ep150_val_accuracy_p30,384.0,4219687.0,99.0,69.0,0.8803571462631226,102.6526579062144,0.8810810810810811,0.88194853436362,0.8410266648149434,0.978786353921489,0.9763779527559056,0.935064935064935,0.889763779527559,0.7346938775510204
efficientnetb0,none,8,aug_tanpa_crop,0,bn1_lriter0_ep150_val_accuracy_p30,224.0,4219687.0,114.0,84.0,0.9303571581840515,44.35024023453395,0.9333333333333333,0.9321180802575627,0.9109377236315377,0.991396802207613,0.968503937007874,0.974025974025974,0.905511811023622,0.8843537414965986
efficientnetb0,none,8,aug_derau_benar,0,bn1_lriter0_ep150_val_accuracy_p30,224.0,4219687.0,56.0,26.0,0.6214285492897034,21.810514346758524,0.6054054054054054,0.5954862843485242,0.4662406956063501,0.8191358926494061,0.6141732283464567,0.7857142857142857,0.30708661417322836,0.6666666666666666
efficientnetb0,none,8,aug_tanpa_flip,0,bn1_lriter0_ep150_val_accuracy_p30,224.0,4219687.0,137.0,107.0,0.8946428298950195,51.46109288533528,0.8918918918918919,0.8927890080392238,0.8554894481259194,0.9827210994778562,0.9763779527559056,0.9285714285714286,0.8740157480314961,0.7959183673469388
efficientnetb0,none,8,tepi,1,bn1_lriter0_ep150_val_accuracy_p30,224.0,4219687.0,150.0,145.0,0.8303571343421936,57.12700047492981,0.8450450450450451,0.8408919565237758,0.7934901590900242,0.9678500121743366,0.8346456692913385,1.0,0.8740157480314961,0.6666666666666666
vit_s16,none,8,dasar,0,bn1_lriter0_ep150_val_accuracy_p30,224.0,21717508.0,93.0,63.0,0.9214285612106323,116.03343371947606,0.9027027027027027,0.9008067443912694,0.8701849982890582,0.9892822552282011,0.9763779527559056,0.961038961038961,0.9448818897637795,0.7414965986394558
efficientnetb0,none,8,aug_tanpa,1,bn1_lriter0_ep150_val_accuracy_p30,224.0,4219687.0,127.0,97.0,0.9428571462631226,47.33363029559453,0.9423423423423424,0.940532162412076,0.9229484370593724,0.9925417309201093,0.952755905511811,1.0,0.9133858267716536,0.8979591836734694
efficientnetb0,none,8,res384,1,bn1_lriter0_ep150_val_accuracy_p30,384.0,4219687.0,97.0,67.0,0.862500011920929,100.72842586437861,0.8378378378378378,0.8399335003807601,0.7828865011475069,0.9664799935070206,0.968503937007874,0.922077922077922,0.8267716535433071,0.6462585034013606
efficientnetb0,none,8,aug_tanpa_crop,1,bn1_lriter0_ep150_val_accuracy_p30,224.0,4219687.0,106.0,76.0,0.9214285612106323,40.04746331771215,0.9279279279279279,0.9279297190569225,0.9035985531098151,0.9909184860536213,0.968503937007874,0.9675324675324676,0.8818897637795275,0.891156462585034
efficientnetb0,none,8,aug_tanpa_flip,1,bn1_lriter0_ep150_val_accuracy_p30,224.0,4219687.0,113.0,83.0,0.887499988079071,43.13100386063258,0.8846846846846846,0.8859042552417461,0.8459242808064682,0.9783415848280714,0.952755905511811,0.8961038961038961,0.8818897637795275,0.8163265306122449
efficientnetb0,none,8,aug_derau_benar,1,bn1_lriter0_ep150_val_accuracy_p30,224.0,4219687.0,150.0,149.0,0.7839285731315613,56.340684191385904,0.781981981981982,0.7858309990662932,0.7074850377649426,0.9428942455969483,0.889763779527559,0.8766233766233766,0.6929133858267716,0.6666666666666666
efficientnetb0,none,8,tepi,2,bn1_lriter0_ep150_val_accuracy_p30,224.0,4219687.0,133.0,103.0,0.8392857313156128,49.723930994669594,0.8306306306306306,0.819133257711005,0.7732931805441484,0.9637421204988773,0.8346456692913385,0.9805194805194806,0.6062992125984252,0.8639455782312925
vit_s16,none,8,dasar,1,bn1_lriter0_ep150_val_accuracy_p30,224.0,21717508.0,108.0,78.0,0.9267857074737549,149.7147933125496,0.9297297297297298,0.9306908042897654,0.9060636571160741,0.9919957795633472,0.968503937007874,0.948051948051948,0.9212598425196851,0.8843537414965986
efficientnetb0,none,8,aug_tanpa,2,bn1_lriter0_ep150_val_accuracy_p30,224.0,4219687.0,111.0,81.0,0.9428571462631226,41.62013347943624,0.9477477477477477,0.9461245316846224,0.9302083559178718,0.9930568406244082,0.937007874015748,1.0,0.9606299212598425,0.891156462585034
efficientnetb0,none,8,aug_tanpa_crop,2,bn1_lriter0_ep150_val_accuracy_p30,224.0,4219687.0,112.0,82.0,0.9303571581840515,43.623137700557706,0.9117117117117117,0.9123720443795876,0.8819707563506635,0.9887022157292427,0.984251968503937,0.9285714285714286,0.8661417322834646,0.8707482993197279
efficientnetb0,none,8,res384,2,bn1_lriter0_ep150_val_accuracy_p30,384.0,4219687.0,144.0,114.0,0.9160714149475098,145.37959849039714,0.9027027027027027,0.9029401957925658,0.8700589222304601,0.983306008711414,0.968503937007874,0.9155844155844156,0.8976377952755905,0.8367346938775511
efficientnetb0,none,8,aug_tanpa_flip,2,bn1_lriter0_ep150_val_accuracy_p30,224.0,4219687.0,150.0,134.0,0.8892857432365417,56.79895188411077,0.8972972972972973,0.899332606724548,0.8626637203870682,0.9799302004707411,0.9606299212598425,0.9155844155844156,0.889763779527559,0.8299319727891157
efficientnetb0,none,8,aug_derau_benar,2,bn1_lriter0_ep150_val_accuracy_p30,224.0,4219687.0,150.0,135.0,0.8125,55.51073435147603,0.8234234234234235,0.8283286345058729,0.763771319866402,0.9559413467521576,0.8582677165354331,0.7727272727272727,0.7952755905511811,0.8707482993197279"""
if not UJI_CEPAT and not os.path.exists(B_CSV):
    open(B_CSV, 'w', encoding='utf-8').write(CSV_SESI1 + chr(10)); print('Hasil sesi 1 ditulis ke', B_CSV)
lama = [B_CSV] + ([] if UJI_CEPAT else glob.glob('/kaggle/input/**/eksperimen_C.csv', recursive=True))
done = [pd.read_csv(f) for f in lama if os.path.exists(f)]
done = pd.concat(done, ignore_index=True) if done else pd.DataFrame(columns=KEY)
if 'varian' not in done.columns: done['varian'] = 'dasar'
if 'konfig' not in done.columns: done['konfig'] = 'lama'
done['konfig'] = done['konfig'].fillna('lama')
done = done.drop_duplicates(KEY, keep='last').reset_index(drop=True)
sudah = set((a, b, int(c), v, int(d)) for a, b, c, v, d in done[done.konfig == TAG][KEY[:5]].values.tolist())
jobs = [dict(backbone=bb, atensi=att, rasio=r, varian=v, seed=sd) for sd, (bb, att, r, v) in itertools.product(RUN_SEEDS, RUN_VARIANTS)]
jobs = [j for j in jobs if (j['backbone'], j['atensi'], j['rasio'], j['varian'], j['seed']) not in sudah]
if UJI_CEPAT: jobs = jobs[:len(RUN_VARIANTS)]      # satu run singkat per varian, supaya semua jalur kode teruji
print(f'Sudah selesai (konfigurasi {TAG}): {len(sudah)} run | antrean: {len(jobs)} run')

# --- unduh bobot ImageNet lebih dulu (di CPU) agar dua pekerja tidak mengunduh berkas yang sama bersamaan
_apps = {'efficientnetb0': keras.applications.EfficientNetB0, 'resnet50': keras.applications.ResNet50, 'mobilenetv2': keras.applications.MobileNetV2}
for bb in sorted({j['backbone'] for j in jobs} & set(_apps)):
    _apps[bb](weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,)); keras.backend.clear_session()
if importlib.util.find_spec('tensorflow_hub') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'tensorflow_hub'], check=False)

slots = list(range(N_GPU)) if N_GPU else [None]            # None = CPU
if N_PARALEL: slots = slots[:N_PARALEL]
print(f'Pekerja paralel: {len(slots)} ->', ['GPU %d' % s if s is not None else 'CPU' for s in slots])

nama_job = lambda j: f"{j['backbone']}_{j['varian']}_s{j['seed']}"
maks_menit = lambda q: MAKS_MENIT.get(q['varian'], MAKS_MENIT.get(q['backbone'], 100))
def ekor(path, n=1):
    try:
        with open(path, encoding='utf-8', errors='replace') as f: return [l.rstrip() for l in f.readlines() if l.strip()][-n:]
    except OSError: return []
menit = lambda: (time.time() - T_MULAI) / 60

aktif, gagal, antre, t_status = {}, [], list(jobs), time.time()
while antre or aktif:
    for s in slots:                                         # isi slot kosong
        muat = [k for k, q in enumerate(antre) if menit() + maks_menit(q) * KONFIG['EPOCHS'] / 150 <= BATAS_JAM * 60]
        if s not in aktif and muat:                         # ambil run pertama di antrean yang masih sempat selesai
            j = antre.pop(muat[0]); nm = nama_job(j)
            jf, rf, lf = f'{LOG_DIR}/{nm}.job.json', f'{LOG_DIR}/{nm}.hasil.json', f'{LOG_DIR}/{nm}.log'
            json.dump(j, open(jf, 'w'))
            if os.path.exists(rf): os.remove(rf)
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=('-1' if s is None else str(s)), PYTHONIOENCODING='utf-8')
            p = subprocess.Popen([sys.executable, 'fundus_lib.py', STATE, jf, rf], stdout=open(lf, 'w', encoding='utf-8'), stderr=subprocess.STDOUT, env=env)
            aktif[s] = (p, j, rf, lf)
            print(f'[{menit():6.1f} mnt] MULAI   {nm} di ' + ('CPU' if s is None else f'GPU {s}'), flush=True)
    if not aktif: break                                     # antrean masih ada tetapi batas waktu tercapai
    time.sleep(5 if UJI_CEPAT else 20)
    for s, (p, j, rf, lf) in list(aktif.items()):           # ambil yang sudah selesai
        if p.poll() is None: continue
        del aktif[s]; nm = nama_job(j)
        if p.returncode == 0 and os.path.exists(rf):
            row = json.load(open(rf)); row['konfig'] = TAG
            done = pd.concat([done, pd.DataFrame([row])], ignore_index=True).drop_duplicates(KEY, keep='last'); done.to_csv(B_CSV, index=False)
            print(f"[{menit():6.1f} mnt] SELESAI {nm}: akurasi uji {row['akurasi']:.4f}, macro-F1 {row['macro_f1']:.4f}, "
                  f"epoch {row['epoch']} (terbaik {row['epoch_terbaik']}), {row['menit']:.1f} mnt", flush=True)
            print('    BARIS_CSV ' + json.dumps(row), flush=True)   # cadangan: CSV bisa disusun ulang dari teks keluaran ini
        else:
            gagal.append(nm); print(f'!! GAGAL {nm} (kode {p.returncode}). Akhir log:\n   ' + '\n   '.join(ekor(lf, 25)), flush=True)
    if time.time() - t_status > (20 if UJI_CEPAT else 600):  # status berkala
        t_status = time.time()
        for s, (p, j, rf, lf) in aktif.items():
            print(f"[{menit():6.1f} mnt]   ...     {nama_job(j)}: {(ekor(lf) or ['memuat'])[0][:110]}", flush=True)

print(f'\nSelesai sesi ini. Run tersimpan untuk konfigurasi {TAG}: {int((done.konfig == TAG).sum())} | belum dijalankan: {len(antre)} | gagal: {gagal}')
if antre: print('Untuk melanjutkan: simpan versi ini, tambahkan keluarannya sebagai Input di sesi berikutnya, lalu jalankan lagi. Run yang selesai dilewati otomatis.')


### C3. Tabel eksperimen tambahan (rata-rata ± SD pada data uji; selisih terhadap model dasar pada seed yang sama)

In [ ]:
fmt = lambda x, d=2: f'{x:.{d}f}'.replace('.', ',')
import io
from scipy import stats
done = pd.read_csv(B_CSV) if os.path.exists(B_CSV) else pd.DataFrame(columns=KEY); done = done[done.konfig == TAG]
assert len(done), 'Belum ada run yang selesai; periksa log di hasil_C/log_C'
dasar = pd.read_csv(io.StringIO(CSV_DASAR)); dasar = dasar[(dasar.backbone == 'efficientnetb0') & (dasar.atensi == 'none')].set_index('seed')
done['nama'] = done.backbone.where(done.varian == 'dasar', done.varian) + done.atensi.map(lambda a: '' if a == 'none' else f'_{a}8')
rows = [dict(Varian='EfficientNetB0 dasar (Bagian B, seed sama)', n=len(sorted(set(done.seed) & set(dasar.index))),
             Akurasi=f"{fmt(100 * dasar.loc[sorted(set(done.seed) & set(dasar.index)), 'akurasi'].mean())} ± {fmt(100 * dasar.loc[sorted(set(done.seed) & set(dasar.index)), 'akurasi'].std())}", Selisih='acuan', p='')]
for nm, g in done.groupby('nama', sort=False):
    g = g.set_index('seed'); sd = sorted(set(g.index) & set(dasar.index)); d = (g.loc[sd, 'akurasi'] - dasar.loc[sd, 'akurasi']) * 100
    p = stats.ttest_rel(g.loc[sd, 'akurasi'], dasar.loc[sd, 'akurasi']).pvalue if len(sd) >= 2 else float('nan')
    rows.append(dict(Varian=nm, n=len(g), Akurasi=f"{fmt(100 * g.akurasi.mean())} ± {fmt(100 * g.akurasi.std() if len(g) > 1 else 0)}",
                     Selisih=f"{d.mean():+.2f} ± {d.std() if len(d) > 1 else 0:.2f}".replace('.', ','), p=fmt(p, 3),
                     Katarak=fmt(100 * g.sens_cataract.mean()), RD=fmt(100 * g.sens_diabetic_retinopathy.mean()), Glaukoma=fmt(100 * g.sens_glaucoma.mean()), Normal=fmt(100 * g.sens_normal.mean()),
                     Epoch=round(g.epoch.mean()), Menit=round(g.menit.mean())))
tabel = pd.DataFrame(rows); display(tabel); tabel.to_csv(f'{OUT_DIR}/tabel_eksperimen_tambahan.csv', index=False)
print('Catatan: varian "tepi" menguji jalan pintas; akurasi acak untuk empat kelas adalah 25%.')
